# SR3 ATM Correlation: Implied vs Realized (Listed-Only, Single-Contract)

## Strategy Overview
Trade cross-tenor ATM correlation between SOFR futures rates around FOMC meetings
using **individual contract ATM straddles** — standard quarterlies AND midcurves.
No packs, bundles, or spread options.

## Core Math
Given two SR3 rates with Bachelier normal vols $\sigma_i$, $\sigma_j$ and pairwise correlation $\rho$:

$$\sigma^2_{\text{spread}} = \sigma^2_i + \sigma^2_j - 2\rho\,\sigma_i\,\sigma_j$$

We observe $\sigma_i$, $\sigma_j$ from listed ATM options and estimate $\rho$ from trailing
realized futures correlation. Since the listed market has no explicit spread option on
single SR3 contracts, **implied $\rho$ is not directly observable**. Instead we:

1. Compute the **market cost** of the straddle pair from individual ATM straddle prices
2. Compute the **model fair value** using $\sigma_{\text{spread}}$ from our $\rho$ estimate
3. Trade the gap: `signal = market_cost - model_value`

## Trade Expression
- Vol-weighted pair of ATM straddles on two SR3 tenors (quarterly or midcurve)
- Entry ~5 business days before each FOMC; exit 1 day after
- Direction based on signal:
  - `signal > 0` : market prices more spread vol than model : **sell correlation** (long both straddles)
  - `signal < 0` : market prices less spread vol : **buy correlation** (short both straddles)

## Pairs (single contracts only — no packs/bundles/spread options)

| Pair | Leg i | Leg j | What it captures |
|------|-------|-------|------------------|
| Near vs Belly | SFRCM2 | SFRCM5 | Quarterly strip front-belly rho |
| Near vs 1Y MC | SFRCM2 | 0QCM1 | Front rate vs 1Y forward rate rho |
| Near vs 2Y MC | SFRCM2 | 2QCM1 | Front rate vs 2Y forward rate rho |

## Correlation Signals
- **Trailing realized rho** (20-day): backward-looking baseline
- **Long-run rho** (120-day): anchor for mean reversion
- **Meeting type**: SEP meetings (Mar/Jun/Sep/Dec) structurally higher rho; interstitials lower
- The model rho conditions on these observables; the signal fires when market-embedded spread vol
  diverges from model expectations

In [ ]:
import sys
sys.path.append("..")

import datetime as dt
import math
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import QuantLib as ql
from scipy.stats import norm as sp_norm
from tqdm.auto import tqdm

from BT.data_handler import TimeGrid
from BT.misc import ql_cal_date_range
from BT.query_actions import AddQueryAction, UnwindPositionsAction
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy
from BT.triggers import DateTrigger, DateTriggerRequirements

from MDP.STIRFutures.STIRFutureMDP import STIRFutureMDP
from MDP.STIRFutures.STIRFutureOptionMDP import STIRFutureOptionMDP
from Query.IRSwaps._CENTRAL_BANK_DATES import _CENTRAL_BANK_DATES

from Query.STIRFutureOptions.STIRFutureOptionQuery import STIRFutureOptionQuery
from Query.STIRFutureOptions.STIRFutureOptionStructure import STIRFutureOptionStructure
from Query.STIRFutureOptions.STIRFutureOptionValue import STIRFutureOptionValue

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
START_DATE = dt.date(2025, 1, 1)
END_DATE = dt.date(2026, 2, 27)

# Cross-tenor pairs: (leg_i_alias, leg_j_alias, label)
# CM aliases are resolved by STIRFutureOptionMDP per as-of date.
# SFRCM<n> = n-th quarterly SR3 contract
# 0QCM<n>  = n-th 1Y midcurve contract (underlying ~1Y further out)
# 2QCM<n>  = n-th 2Y midcurve contract (underlying ~2Y further out)
PAIRS = [
    ("SFRCM2", "SFRCM5", "NearQ_vs_BellyQ"),
    ("SFRCM2", "0QCM1", "NearQ_vs_1YMC"),
    ("SFRCM2", "2QCM1", "NearQ_vs_2YMC"),
]

# Timing around FOMC events
ENTRY_BDAYS_BEFORE_EVENT = 5
EXIT_BDAYS_AFTER_EVENT = 1

# Realized correlation parameters
REALIZED_LOOKBACK = 20        # trailing bdays for short-term rho
LONGRUN_LOOKBACK = 120        # trailing bdays for long-run rho anchor

# Signal threshold
SIGNAL_THRESHOLD = 0.0        # minimum |signal| to trade (in straddle price units)

# Event-window attribution (wider window needed for meaningful event-rho)
EVENT_WINDOW_PRE_BDAYS = 3
EVENT_WINDOW_POST_BDAYS = 3

# SEP meeting months (Summary of Economic Projections with dot plots)
SEP_MONTHS = {3, 6, 9, 12}

# Execution / runtime safety
MAX_EVENTS = None             # None = all FOMC in range
MAX_TRADES = None
REQUEST_SLEEP_SECONDS = 0.25
REQUEST_RETRIES = 3

POINT_VALUE = 2500.0          # $2,500 per IMM index point for SR3

# Optional
RUN_DELTA_HEDGE_ATTRIBUTION = False

In [ ]:
# ---------------------------------------------------------------------------
# MDPs and calendar
# ---------------------------------------------------------------------------
stirf_mdp = STIRFutureMDP(source="BARCHART_TOS_LIVE_STIRF-RL")
stirfo_mdp = STIRFutureOptionMDP(source="BARCHART_STIRFO-QL")


class FlatOptionMDP:
    """Flatten dict[str, list[pricer]] -> dict[str, pricer] for BT compatibility."""

    def __init__(self, inner):
        self.inner = inner

    @staticmethod
    def _flatten(out):
        if not isinstance(out, dict):
            return out
        return {k: (v[0] if isinstance(v, list) and len(v) == 1 else v) for k, v in out.items()}

    def get_pricer(self, request):
        return self._flatten(self.inner.get_pricer(request))

    def get_data(self, request):
        return self._flatten(self.inner.get_data(request))


flat_stirfo_mdp = FlatOptionMDP(stirfo_mdp)

# Business-day calendar — extend back far enough for LONGRUN_LOOKBACK
CAL = ql.UnitedStates(ql.UnitedStates.GovernmentBond)

_cal_start = START_DATE - dt.timedelta(days=int(LONGRUN_LOOKBACK * 2.0))
ALL_BDAYS = [
    pd.Timestamp(x).date()
    for x in ql_cal_date_range(
        ql_cal=CAL,
        start=dt.datetime.combine(_cal_start, dt.time()),
        end=dt.datetime.combine(END_DATE, dt.time()),
    )
]
BDAY_TO_INDEX = {d: i for i, d in enumerate(ALL_BDAYS)}


def nearest_prev_bday(d: dt.date) -> Optional[dt.date]:
    x = d
    while x not in BDAY_TO_INDEX:
        x -= dt.timedelta(days=1)
        if x < ALL_BDAYS[0]:
            return None
    return x


def shift_bday(d: dt.date, offset: int) -> Optional[dt.date]:
    d0 = nearest_prev_bday(d)
    if d0 is None:
        return None
    i = BDAY_TO_INDEX[d0] + int(offset)
    if 0 <= i < len(ALL_BDAYS):
        return ALL_BDAYS[i]
    return None


def bdays_between(start: dt.date, end: dt.date) -> List[dt.date]:
    s, e = nearest_prev_bday(start), nearest_prev_bday(end)
    if s is None or e is None:
        return []
    i0, i1 = BDAY_TO_INDEX[s], BDAY_TO_INDEX[e]
    return ALL_BDAYS[i0 : i1 + 1] if i1 >= i0 else []


print(f"Calendar: {ALL_BDAYS[0]} to {ALL_BDAYS[-1]}  ({len(ALL_BDAYS)} bdays)")

In [ ]:
# ---------------------------------------------------------------------------
# Data fetching (with caching), realized correlation, Bachelier utilities
# ---------------------------------------------------------------------------
_option_cache: Dict[Tuple[dt.date, str], object] = {}
_future_cache: Dict[Tuple[dt.date, str], object] = {}


def _safe_first(v):
    if isinstance(v, list):
        return v[0] if v else None
    return v


def _retry_fetch(fetch_fn, retries=REQUEST_RETRIES):
    for i in range(retries):
        try:
            return fetch_fn()
        except Exception:
            if i == retries - 1:
                return {}
            time.sleep((i + 1) * 0.5)
    return {}


def get_option_pricers(as_of: dt.date, symbols: List[str]) -> Dict[str, object]:
    symbols = [str(s).upper() for s in symbols]
    missing = [s for s in symbols if (as_of, s) not in _option_cache]
    if missing:
        time.sleep(REQUEST_SLEEP_SECONDS)
        out = _retry_fetch(
            lambda: stirfo_mdp.get_data(
                {"endpoint": "option_snapshot", "symbols": missing,
                 "timestamp": as_of, "show_tqdm": False}
            )
        )
        for s in missing:
            _option_cache[(as_of, s)] = _safe_first(out.get(s))
    return {s: _option_cache.get((as_of, s)) for s in symbols}


def get_future_pricers(as_of: dt.date, symbols: List[str]) -> Dict[str, object]:
    symbols = [str(s).upper() for s in symbols]
    missing = [s for s in symbols if (as_of, s) not in _future_cache]
    if missing:
        time.sleep(REQUEST_SLEEP_SECONDS)
        out = _retry_fetch(
            lambda: stirf_mdp.get_data(
                {"symbols": missing, "timestamp": as_of, "show_tqdm": False}
            )
        )
        for s in missing:
            _future_cache[(as_of, s)] = _safe_first(out.get(s))
    return {s: _future_cache.get((as_of, s)) for s in symbols}


def _pricer_field(pr, field: str) -> float:
    """Safely extract a float field from a pricer."""
    if pr is None:
        return np.nan
    try:
        v = getattr(pr, field)()
        return float(v) if np.isfinite(v) else np.nan
    except Exception:
        return np.nan


def fetch_atm_leg(as_of: dt.date, cm_alias: str) -> Optional[dict]:
    """Fetch ATM call for a CM alias; return vol / price / underlying info.
    Straddle price approximated as 2 * ATM_call (exact in Bachelier at ATM)."""
    sym = f"{cm_alias}|ATMC"
    pmap = get_option_pricers(as_of, [sym])
    pr = pmap.get(sym.upper())
    if pr is None:
        return None
    iv = _pricer_field(pr, "iv_normal")
    px = _pricer_field(pr, "price")
    if not (np.isfinite(iv) and iv > 0 and np.isfinite(px)):
        return None
    try:
        return {
            "symbol": pr.symbol(),
            "underlying_symbol": pr.underlying_symbol(),
            "call_price": px,
            "straddle_price": 2.0 * px,
            "iv_normal": iv,
            "iv_normal_bps": _pricer_field(pr, "iv_normal_bps"),
            "delta": _pricer_field(pr, "delta"),
            "vega": _pricer_field(pr, "vega"),
            "expiry_date": pr.expiry_date(),
        }
    except Exception:
        return None


# --- Realized correlation ---

def realized_corr(sym_i: str, sym_j: str, end_date: dt.date, lookback: int) -> float:
    """Trailing pairwise correlation of daily futures price changes."""
    end_bday = nearest_prev_bday(end_date)
    if end_bday is None:
        return np.nan
    i_end = BDAY_TO_INDEX[end_bday]
    i_start = i_end - lookback
    if i_start < 1:
        return np.nan
    dates = ALL_BDAYS[i_start - 1 : i_end + 1]
    rows = []
    for d in dates:
        pmap = get_future_pricers(d, [sym_i, sym_j])
        pi, pj = pmap.get(sym_i), pmap.get(sym_j)
        if pi is None or pj is None:
            continue
        rows.append((d, float(pi.price()), float(pj.price())))
    if len(rows) < max(10, lookback // 2):
        return np.nan
    df = pd.DataFrame(rows, columns=["date", "i", "j"]).set_index("date").sort_index()
    c = df["i"].diff().corr(df["j"].diff())
    return float(c) if np.isfinite(c) else np.nan


def event_window_corr(sym_i: str, sym_j: str, event_date: dt.date,
                       pre_bdays: int, post_bdays: int) -> float:
    """Realized correlation in a window around an event."""
    d0 = nearest_prev_bday(event_date)
    if d0 is None:
        return np.nan
    i0 = BDAY_TO_INDEX[d0]
    dates = ALL_BDAYS[max(0, i0 - pre_bdays) : min(len(ALL_BDAYS), i0 + post_bdays + 1)]
    if len(dates) < 3:
        return np.nan
    rows = []
    for d in dates:
        pmap = get_future_pricers(d, [sym_i, sym_j])
        pi, pj = pmap.get(sym_i), pmap.get(sym_j)
        if pi is None or pj is None:
            continue
        rows.append((d, float(pi.price()), float(pj.price())))
    if len(rows) < 3:
        return np.nan
    df = pd.DataFrame(rows, columns=["date", "i", "j"]).set_index("date").sort_index()
    c = df["i"].diff().corr(df["j"].diff())
    return float(c) if np.isfinite(c) else np.nan


# --- Bachelier spread-vol utilities ---

def spread_vol(sigma_i: float, sigma_j: float, rho: float) -> float:
    """sigma_spread = sqrt(sigma_i^2 + sigma_j^2 - 2*rho*sigma_i*sigma_j)"""
    v = sigma_i**2 + sigma_j**2 - 2.0 * rho * sigma_i * sigma_j
    return math.sqrt(max(v, 0.0))


def bachelier_atm_straddle(sigma: float, T: float) -> float:
    """ATM Bachelier straddle price = sigma * sqrt(T) * sqrt(2/pi)"""
    return sigma * math.sqrt(max(T, 1e-6)) * math.sqrt(2.0 / math.pi)


def implied_rho(sigma_i: float, sigma_j: float, sigma_sp: float) -> float:
    """Back out rho from individual vols and observed spread vol."""
    denom = 2.0 * sigma_i * sigma_j
    if denom < 1e-12:
        return np.nan
    rho = (sigma_i**2 + sigma_j**2 - sigma_sp**2) / denom
    return float(np.clip(rho, -1.0, 1.0))

In [ ]:
# ---------------------------------------------------------------------------
# FOMC events + signal generation (per pair x per meeting)
# ---------------------------------------------------------------------------
fomc_events = sorted(
    {v[0] for v in _CENTRAL_BANK_DATES["USD-FEDFUNDS"].values()
     if START_DATE <= v[0] <= END_DATE}
)
if MAX_EVENTS is not None:
    fomc_events = fomc_events[:MAX_EVENTS]

print(f"FOMC meetings in range: {len(fomc_events)}")
for ev in fomc_events:
    print(f"  {ev}  {'SEP' if ev.month in SEP_MONTHS else ''}")

# --- Main loop ---
rows = []
for pair_i_alias, pair_j_alias, pair_label in PAIRS:
    for ev in tqdm(fomc_events, desc=f"Signals: {pair_label}"):
        entry_date = shift_bday(ev, -ENTRY_BDAYS_BEFORE_EVENT)
        exit_date = shift_bday(ev, EXIT_BDAYS_AFTER_EVENT)
        if entry_date is None or exit_date is None or entry_date >= exit_date:
            continue

        # --- Fetch ATM option data on entry date ---
        leg_i = fetch_atm_leg(entry_date, pair_i_alias)
        leg_j = fetch_atm_leg(entry_date, pair_j_alias)
        if leg_i is None or leg_j is None:
            continue

        sigma_i = leg_i["iv_normal"]
        sigma_j = leg_j["iv_normal"]
        und_i = str(leg_i["underlying_symbol"])
        und_j = str(leg_j["underlying_symbol"])

        # --- Trailing realized rho ---
        rho_trail = realized_corr(und_i, und_j, entry_date, REALIZED_LOOKBACK)
        rho_longrun = realized_corr(und_i, und_j, entry_date, LONGRUN_LOOKBACK)
        if not np.isfinite(rho_trail):
            continue

        # --- Model rho: conditional estimate ---
        # v1: long-run anchor with SEP adjustment
        is_sep = ev.month in SEP_MONTHS
        rho_base = rho_longrun if np.isfinite(rho_longrun) else rho_trail
        # SEP meetings: structurally higher rho (dots + action reinforce)
        # Interstitial: structurally lower rho (no dots, noisier belly response)
        sep_adj = 0.03 if is_sep else -0.02
        rho_model = float(np.clip(rho_base + sep_adj, -0.999, 0.999))

        # --- Spread vol from model vs from trailing ---
        sigma_sp_model = spread_vol(sigma_i, sigma_j, rho_model)
        sigma_sp_trail = spread_vol(sigma_i, sigma_j, rho_trail)

        # --- Time to expiry (approximate) ---
        exp_i = leg_i.get("expiry_date")
        exp_j = leg_j.get("expiry_date")
        T_i = max((exp_i - entry_date).days, 1) / 365.0 if exp_i else 0.1
        T_j = max((exp_j - entry_date).days, 1) / 365.0 if exp_j else 0.1
        T_avg = (T_i + T_j) / 2.0

        # --- Model spread straddle value vs market cost ---
        # Market cost: sum of individual ATM straddles at vol-weighted ratio
        # Vol-weighted ratio: w_i/w_j = sigma_j/sigma_i (vega-match)
        ratio_i = sigma_j / (sigma_i + sigma_j)
        ratio_j = sigma_i / (sigma_i + sigma_j)
        market_cost = ratio_i * leg_i["straddle_price"] + ratio_j * leg_j["straddle_price"]

        # Model fair value of a spread straddle using model rho
        model_value = bachelier_atm_straddle(sigma_sp_model, T_avg)

        # Signal: market cost - model value
        # > 0: market embeds more spread vol than model -> sell corr (long both straddles)
        # < 0: market embeds less spread vol -> buy corr (short both straddles)
        signal = market_cost - model_value
        trade_side = np.sign(signal) if abs(signal) > SIGNAL_THRESHOLD else 0.0

        # --- Weights for the straddle pair ---
        weight_i = trade_side * ratio_i
        weight_j = trade_side * ratio_j

        # --- Event-window realized rho (for attribution) ---
        rho_event = event_window_corr(
            und_i, und_j, ev, EVENT_WINDOW_PRE_BDAYS, EVENT_WINDOW_POST_BDAYS
        )

        rows.append({
            "pair": pair_label,
            "event_date": ev,
            "is_sep": is_sep,
            "entry_date": entry_date,
            "exit_date": exit_date,
            "symbol_i": str(leg_i["symbol"]),
            "symbol_j": str(leg_j["symbol"]),
            "underlying_i": und_i,
            "underlying_j": und_j,
            "vol_i_bps": leg_i["iv_normal_bps"],
            "vol_j_bps": leg_j["iv_normal_bps"],
            "sigma_i": sigma_i,
            "sigma_j": sigma_j,
            "T_avg": T_avg,
            "straddle_px_i": leg_i["straddle_price"],
            "straddle_px_j": leg_j["straddle_price"],
            "rho_trail": rho_trail,
            "rho_longrun": rho_longrun if np.isfinite(rho_longrun) else np.nan,
            "rho_model": rho_model,
            "rho_event": rho_event if np.isfinite(rho_event) else np.nan,
            "sigma_sp_model": sigma_sp_model,
            "sigma_sp_trail": sigma_sp_trail,
            "market_cost": market_cost,
            "model_value": model_value,
            "signal": signal,
            "trade_side": trade_side,
            "ratio_i": ratio_i,
            "ratio_j": ratio_j,
            "weight_i": weight_i,
            "weight_j": weight_j,
        })

signals_df = pd.DataFrame(rows)
if signals_df.empty:
    print("No valid signals produced (likely data gaps or throttling).")
else:
    signals_df = signals_df.sort_values(["pair", "entry_date"]).reset_index(drop=True)
    signals_df["signal_abs"] = signals_df["signal"].abs()
    signals_df["tag"] = [
        f"sr3corr_{r['pair']}_{r['entry_date']:%Y%m%d}"
        for _, r in signals_df.iterrows()
    ]

print(f"\nTotal signals: {len(signals_df)}")
for p in signals_df["pair"].unique():
    n = (signals_df["pair"] == p).sum()
    print(f"  {p}: {n}")

In [ ]:
# ---------------------------------------------------------------------------
# Signal summary & trade selection
# ---------------------------------------------------------------------------
if signals_df.empty:
    trade_df = signals_df.copy()
else:
    # Filter: only trade when signal is nonzero (trade_side != 0)
    trade_df = signals_df[signals_df["trade_side"] != 0.0].copy()
    if MAX_TRADES is not None:
        trade_df = trade_df.head(MAX_TRADES)

print(f"Signals: {len(signals_df)} | Trades selected: {len(trade_df)}")
print(f"  Long straddle pair (sell corr): {(trade_df['trade_side'] > 0).sum()}")
print(f"  Short straddle pair (buy corr): {(trade_df['trade_side'] < 0).sum()}")

display_cols = [
    "pair", "event_date", "is_sep", "entry_date", "exit_date",
    "underlying_i", "underlying_j",
    "vol_i_bps", "vol_j_bps", "rho_trail", "rho_model", "rho_event",
    "sigma_sp_model", "market_cost", "model_value", "signal", "trade_side",
]
trade_df[[c for c in display_cols if c in trade_df.columns]] if not trade_df.empty else trade_df

In [ ]:
# ---------------------------------------------------------------------------
# P&L computation: entry/exit ATM call marks -> straddle P&L
# ---------------------------------------------------------------------------
attr_rows = []

if not trade_df.empty:
    for r in tqdm(trade_df.to_dict(orient="records"), desc="P&L Calc"):
        # Fetch ATM call on entry and exit for each leg
        entry_i = fetch_atm_leg(r["entry_date"], r["symbol_i"].rsplit("|", 1)[0])
        entry_j = fetch_atm_leg(r["entry_date"], r["symbol_j"].rsplit("|", 1)[0])

        # For exit, fetch using the RESOLVED symbol (not CM alias, which may roll)
        exit_pricers = get_option_pricers(r["exit_date"], [r["symbol_i"], r["symbol_j"]])
        p_exit_i = exit_pricers.get(r["symbol_i"])
        p_exit_j = exit_pricers.get(r["symbol_j"])

        if entry_i is None or entry_j is None or p_exit_i is None or p_exit_j is None:
            continue

        # Straddle P&L = 2 * (exit_call - entry_call) per unit
        exit_call_i = _pricer_field(p_exit_i, "price")
        exit_call_j = _pricer_field(p_exit_j, "price")
        if not (np.isfinite(exit_call_i) and np.isfinite(exit_call_j)):
            continue

        straddle_pnl_i = 2.0 * (exit_call_i - entry_i["call_price"])
        straddle_pnl_j = 2.0 * (exit_call_j - entry_j["call_price"])

        # Weighted P&L
        unhedged_pnl = POINT_VALUE * (
            r["weight_i"] * straddle_pnl_i + r["weight_j"] * straddle_pnl_j
        )

        # Exit vols for attribution
        exit_vol_i = _pricer_field(p_exit_i, "iv_normal")
        exit_vol_j = _pricer_field(p_exit_j, "iv_normal")

        attr_rows.append({
            **r,
            "straddle_pnl_i": straddle_pnl_i,
            "straddle_pnl_j": straddle_pnl_j,
            "unhedged_pnl": float(unhedged_pnl),
            "exit_vol_i": exit_vol_i,
            "exit_vol_j": exit_vol_j,
            "vol_chg_i": exit_vol_i - r["sigma_i"] if np.isfinite(exit_vol_i) else np.nan,
            "vol_chg_j": exit_vol_j - r["sigma_j"] if np.isfinite(exit_vol_j) else np.nan,
        })

attr_df = pd.DataFrame(attr_rows)
print(f"Trades with P&L: {len(attr_df)}")

attr_df[[
    "pair", "event_date", "is_sep", "trade_side",
    "rho_trail", "rho_model", "rho_event",
    "signal", "unhedged_pnl",
]] if not attr_df.empty else attr_df

In [ ]:
# ---------------------------------------------------------------------------
# P&L summary and cumulative plot
# ---------------------------------------------------------------------------
if not attr_df.empty:
    print("=== P&L Summary ===")
    for p in attr_df["pair"].unique():
        sub = attr_df[attr_df["pair"] == p]
        total = sub["unhedged_pnl"].sum()
        avg = sub["unhedged_pnl"].mean()
        std = sub["unhedged_pnl"].std()
        hit = (sub["unhedged_pnl"] > 0).mean()
        n = len(sub)
        sharpe = avg / std * np.sqrt(8) if std > 0 else np.nan  # annualize ~8 FOMC/yr
        print(f"  {p}: N={n}  Total=${total:,.0f}  Avg=${avg:,.0f}  Hit={hit:.0%}  Sharpe={sharpe:.2f}")

    print(f"\n  ALL: N={len(attr_df)}  Total=${attr_df['unhedged_pnl'].sum():,.0f}")

    # Cumulative P&L by pair
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Per-pair cumulative
    ax = axes[0]
    for p in attr_df["pair"].unique():
        sub = attr_df[attr_df["pair"] == p].sort_values("event_date")
        cum = sub["unhedged_pnl"].cumsum()
        ax.plot(sub["event_date"].values, cum.values, "o-", label=p, lw=1.5, ms=4)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title("Cumulative P&L by Pair")
    ax.set_ylabel("P&L ($)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Total cumulative
    ax = axes[1]
    total_cum = attr_df.sort_values("event_date")["unhedged_pnl"].cumsum()
    ax.plot(attr_df.sort_values("event_date")["event_date"].values, total_cum.values,
            "o-", color="navy", lw=1.5, ms=4)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title("Cumulative P&L (All Pairs)")
    ax.set_ylabel("P&L ($)")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No attribution data.")

In [ ]:
# ---------------------------------------------------------------------------
# Attribution scatter plots
# ---------------------------------------------------------------------------
if not attr_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))

    # 1. Signal vs P&L
    ax = axes[0, 0]
    for p in attr_df["pair"].unique():
        sub = attr_df[attr_df["pair"] == p]
        ax.scatter(sub["signal"], sub["unhedged_pnl"], label=p, alpha=0.7, s=40)
    ax.axvline(0, color="k", lw=0.5)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xlabel("Signal (market_cost - model_value)")
    ax.set_ylabel("Unhedged P&L ($)")
    ax.set_title("P&L vs Signal")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    # 2. Trailing rho vs Event rho
    ax = axes[0, 1]
    valid = attr_df.dropna(subset=["rho_trail", "rho_event"])
    for p in valid["pair"].unique():
        sub = valid[valid["pair"] == p]
        ax.scatter(sub["rho_trail"], sub["rho_event"], label=p, alpha=0.7, s=40)
    lims = [valid[["rho_trail", "rho_event"]].min().min() - 0.05,
            valid[["rho_trail", "rho_event"]].max().max() + 0.05] if not valid.empty else [0, 1]
    ax.plot(lims, lims, "k--", lw=0.8, alpha=0.5)
    ax.set_xlabel("Trailing rho (pre-event)")
    ax.set_ylabel("Event-window rho")
    ax.set_title("Trailing vs Event Realized Correlation")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    # 3. Rho change vs P&L
    ax = axes[1, 0]
    valid2 = attr_df.dropna(subset=["rho_event"])
    if not valid2.empty:
        valid2 = valid2.copy()
        valid2["rho_change"] = valid2["rho_event"] - valid2["rho_trail"]
        for p in valid2["pair"].unique():
            sub = valid2[valid2["pair"] == p]
            ax.scatter(sub["rho_change"], sub["unhedged_pnl"], label=p, alpha=0.7, s=40)
    ax.axvline(0, color="k", lw=0.5)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xlabel("rho_event - rho_trail")
    ax.set_ylabel("Unhedged P&L ($)")
    ax.set_title("P&L vs Correlation Surprise")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    # 4. SEP vs Interstitial P&L distribution
    ax = axes[1, 1]
    sep_pnl = attr_df[attr_df["is_sep"]]["unhedged_pnl"]
    non_sep_pnl = attr_df[~attr_df["is_sep"]]["unhedged_pnl"]
    x_pos = [0, 1]
    bp = ax.boxplot([sep_pnl.values if len(sep_pnl) else [0],
                     non_sep_pnl.values if len(non_sep_pnl) else [0]],
                    positions=x_pos, widths=0.4, patch_artist=True)
    bp["boxes"][0].set_facecolor("steelblue")
    bp["boxes"][1].set_facecolor("coral")
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"SEP (n={len(sep_pnl)})", f"Interstitial (n={len(non_sep_pnl)})"])
    ax.axhline(0, color="k", lw=0.5)
    ax.set_ylabel("Unhedged P&L ($)")
    ax.set_title("P&L by Meeting Type")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Correlation regime analysis: rho distributions and signal quality
# ---------------------------------------------------------------------------
if not signals_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # 1. Distribution of trailing rho by pair
    ax = axes[0]
    for p in signals_df["pair"].unique():
        sub = signals_df[signals_df["pair"] == p]
        ax.hist(sub["rho_trail"].dropna(), bins=15, alpha=0.5, label=p, density=True)
    ax.set_xlabel("Trailing rho (20-day)")
    ax.set_title("Distribution of Trailing Correlation")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    # 2. SEP vs interstitial rho levels
    ax = axes[1]
    sep = signals_df[signals_df["is_sep"]]["rho_trail"].dropna()
    non_sep = signals_df[~signals_df["is_sep"]]["rho_trail"].dropna()
    bp = ax.boxplot([sep.values if len(sep) else [0],
                     non_sep.values if len(non_sep) else [0]],
                    positions=[0, 1], widths=0.4, patch_artist=True)
    bp["boxes"][0].set_facecolor("steelblue")
    bp["boxes"][1].set_facecolor("coral")
    ax.set_xticks([0, 1])
    ax.set_xticklabels([f"SEP (n={len(sep)})", f"Interstitial (n={len(non_sep)})"])
    ax.set_ylabel("Trailing rho")
    ax.set_title("Correlation by Meeting Type")
    ax.grid(alpha=0.3)

    # 3. Signal distribution
    ax = axes[2]
    for p in signals_df["pair"].unique():
        sub = signals_df[signals_df["pair"] == p]
        ax.hist(sub["signal"].dropna(), bins=15, alpha=0.5, label=p, density=True)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_xlabel("Signal (market_cost - model_value)")
    ax.set_title("Signal Distribution")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary stats
    print("\n=== Correlation Summary ===")
    for p in signals_df["pair"].unique():
        sub = signals_df[signals_df["pair"] == p]
        rt = sub["rho_trail"].dropna()
        re = sub["rho_event"].dropna()
        print(f"  {p}:")
        print(f"    Trail rho: mean={rt.mean():.3f} std={rt.std():.3f} min={rt.min():.3f} max={rt.max():.3f}")
        if len(re):
            print(f"    Event rho: mean={re.mean():.3f} std={re.std():.3f}")

In [ ]:
# ---------------------------------------------------------------------------
# BT engine integration (optional — for proper portfolio tracking)
# ---------------------------------------------------------------------------
triggers = []

if not trade_df.empty:
    for r in trade_df.itertuples(index=False):
        q_i = STIRFutureOptionQuery(
            structure=STIRFutureOptionStructure.OUTRIGHT,
            value=STIRFutureOptionValue.PRICE,
            symbol=r.symbol_i,
            structure_kwargs={"symbol": r.symbol_i, "risk_weights": [float(r.weight_i)]},
            tags=(r.tag, "sr3_corr", "leg_i"),
        )
        q_j = STIRFutureOptionQuery(
            structure=STIRFutureOptionStructure.OUTRIGHT,
            value=STIRFutureOptionValue.PRICE,
            symbol=r.symbol_j,
            structure_kwargs={"symbol": r.symbol_j, "risk_weights": [float(r.weight_j)]},
            tags=(r.tag, "sr3_corr", "leg_j"),
        )

        enter = DateTrigger(
            DateTriggerRequirements(dates=[r.entry_date]),
            actions=[
                AddQueryAction(query=q_i, meta={"trade_tag": r.tag, "leg": "i"}),
                AddQueryAction(query=q_j, meta={"trade_tag": r.tag, "leg": "j"}),
            ],
        )
        exit_ = DateTrigger(
            DateTriggerRequirements(dates=[r.exit_date]),
            actions=[UnwindPositionsAction(match_tag=r.tag, fee=0.0)],
        )
        triggers.extend([enter, exit_])

print(f"Built triggers: {len(triggers)}")

# --- Run ---
bt = None
mtm = pd.Series(dtype=float)

if triggers:
    tg = TimeGrid(
        ql_cal_date_range(
            ql_cal=CAL,
            start=dt.datetime.combine(START_DATE, dt.time()),
            end=dt.datetime.combine(END_DATE, dt.time()),
        )
    )
    strategy = QueryStrategy(
        name="SR3 ATM Correlation",
        triggers=triggers,
        mdps={"STIRFUTUREOPTION": flat_stirfo_mdp},
    )
    bt = QueryDrivenBacktest(
        time_grid=tg, strategy=strategy,
        show_progress=True, progress_desc="BT SR3 Corr",
    )
    bt.run()
    mtm = pd.Series(bt.mtm_history).sort_index()
    print(f"Final MTM: ${float(mtm.iloc[-1]):,.0f}" if len(mtm) else "Empty MTM")
else:
    print("No triggers to run.")

if len(mtm):
    plt.figure(figsize=(12, 5))
    plt.plot(mtm.index, mtm.values, lw=1.6)
    plt.title("SR3 Corr Strategy MTM (BT Engine)")
    plt.ylabel("PnL ($)")
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Delta-hedge overlay (optional, API intensive)
# ---------------------------------------------------------------------------

def trade_delta_hedge_pnl(row) -> float:
    """Compute cumulative delta-hedge P&L for a single trade."""
    dts = bdays_between(row["entry_date"], row["exit_date"])
    if len(dts) < 2:
        return 0.0

    hedge = 0.0
    for d0, d1 in zip(dts[:-1], dts[1:]):
        opt0 = get_option_pricers(d0, [row["symbol_i"], row["symbol_j"]])
        fut0 = get_future_pricers(d0, [row["underlying_i"], row["underlying_j"]])
        fut1 = get_future_pricers(d1, [row["underlying_i"], row["underlying_j"]])

        oi, oj = opt0.get(row["symbol_i"]), opt0.get(row["symbol_j"])
        f0i, f0j = fut0.get(row["underlying_i"]), fut0.get(row["underlying_j"])
        f1i, f1j = fut1.get(row["underlying_i"]), fut1.get(row["underlying_j"])

        if any(x is None for x in [oi, oj, f0i, f0j, f1i, f1j]):
            continue

        di = _pricer_field(oi, "delta")
        dj = _pricer_field(oj, "delta")
        dfi = _pricer_field(f1i, "price") - _pricer_field(f0i, "price")
        dfj = _pricer_field(f1j, "price") - _pricer_field(f0j, "price")

        if any(not np.isfinite(x) for x in [di, dj, dfi, dfj]):
            continue

        # Delta-hedge P&L offsets the straddle's delta exposure
        # Straddle delta = 2 * call_delta (call delta for the straddle position)
        hedge += -POINT_VALUE * (
            row["weight_i"] * 2.0 * di * dfi
            + row["weight_j"] * 2.0 * dj * dfj
        )

    return float(hedge)


if RUN_DELTA_HEDGE_ATTRIBUTION and not attr_df.empty:
    attr_df = attr_df.copy()
    attr_df["delta_hedge_pnl"] = [
        trade_delta_hedge_pnl(r) for _, r in tqdm(attr_df.iterrows(), desc="Delta Hedge", total=len(attr_df))
    ]
    attr_df["hedged_pnl"] = attr_df["unhedged_pnl"] + attr_df["delta_hedge_pnl"]

    print("\n=== Delta-Hedged P&L ===")
    for p in attr_df["pair"].unique():
        sub = attr_df[attr_df["pair"] == p]
        print(f"  {p}: Unhedged=${sub['unhedged_pnl'].sum():,.0f}  "
              f"Hedge=${sub['delta_hedge_pnl'].sum():,.0f}  "
              f"Net=${sub['hedged_pnl'].sum():,.0f}")

    display(attr_df[[
        "pair", "event_date", "signal", "unhedged_pnl", "delta_hedge_pnl", "hedged_pnl"
    ]])
else:
    print("Delta-hedge overlay skipped. Set RUN_DELTA_HEDGE_ATTRIBUTION=True to run.")

## Interpretation & Risk Notes

### Signal Logic
- `signal = market_cost - model_value` where:
  - `market_cost` = vol-weighted sum of listed ATM straddle prices
  - `model_value` = Bachelier ATM straddle on the spread using `rho_model`
- `signal > 0` means the market prices more spread vol than the model expects.
  This implies **lower embedded correlation** than our model. We **sell correlation**
  (long both straddles), profiting if the spread does move more than model.
- `signal < 0` means the market prices less spread vol. We **buy correlation**
  (short both straddles), profiting if rates move together.

### What's Being Traded
- **ATM correlation** between two points on the SOFR strip, realized around FOMC meetings.
- This is an **RV trade**: no directional view on rates needed. Edge comes from
  a better estimate of cross-tenor correlation than what's embedded in the
  listed surface.
- The straddle pair is NOT a pure correlation swap — residual vol-ratio and
  forward vol effects leak in. See context discussion on forward vol contamination.

### Midcurve Pairs
- `0QCM1` = 1Y midcurve: option expiring soon, underlying ~1Y further out on the strip.
  Its vol directly prices **forward vol** at the 1Y point.
- `2QCM1` = 2Y midcurve: same concept, underlying ~2Y out.
- The front-vs-midcurve pairs capture the correlation between **spot rate vol**
  and **forward rate vol** — which is driven by how persistent rate shocks are
  (mean reversion speed).

### Correlation Drivers (for forming views on rho)
1. **Meeting type**: SEP meetings (with dots) are structurally higher-rho
2. **Pricing certainty**: 50/50 meetings are high-rho (action is informative)
3. **Dot dispersion**: clustered dots -> high rho; dispersed -> low rho
4. **Communication regime**: forward guidance -> low rho; data-dependent -> high rho
5. **Regime transitions**: first cut/hike in a cycle -> very high rho
6. **Positioning**: dealer gamma imbalances distort implied ρ from fundamentals

### Caveats
- Straddle price approximated as `2 * ATM_call` (exact in Bachelier at-the-money)
- Implied rho is **not directly observable** from single-contract options — the
  signal compares model fair value to market cost, not implied vs realized rho
- Event-window rho needs 5+ data points to be statistically meaningful; single-day
  event rho is undefined
- If data is sparse, reduce `MAX_EVENTS` or increase `REQUEST_SLEEP_SECONDS`